# 85일차 · 시험 배포 모니터링과 실패 사례 재평가

시험 배포에서 고객이 구판 안내를 받았다고 표시했습니다. 이 실행을 재현 가능한 질문으로 바꾸고, 개선에 쓰지 않은 질문으로 마지막 검사를 합니다.

시험 요청의 지연·오류·피드백을 살펴본다. 실패한 실행 한 건에 기대 동작을 붙여 평가 데이터에 추가하고, 수정 전후 결과와 별도로 보관한 질문의 최종 평가 결과를 제출한다.

## 시험 요청에서 재현 사례 만들기

In [ ]:
runs = [
    {"id":"r1", "as_of":"2026-07-01", "days":30, "ms":460, "feedback":"구판 안내"},
    {"id":"r2", "as_of":"2026-06-30", "days":30, "ms":190, "feedback":""},
]
failed = next(run for run in runs if run["feedback"])
regression = {"as_of":failed["as_of"], "expected":90, "source_run":failed["id"]}
print(regression)

{'as_of': '2026-07-01', 'expected': 90, 'source_run': 'r1'}


기대값 90은 7월 개정 상품설명서에서 확인한 값입니다. 피드백은 다시 볼 실행을 찾는 단서로 쓰고, 정답은 해당 날짜의 원문에서 확인합니다.

### 직접 해 보기

기존 함수와 날짜 조건을 고친 함수에 같은 실패 사례를 넣으세요. 연습용 확인 질문 두 건을 마지막에 실행하세요. 팀 서비스에서는 다른 팀이 따로 보관한 질문을 수정 종료 후 한 번만 열어 평가합니다.

In [ ]:
# 회귀 사례와 최종 확인용 사례의 결과를 따로 출력하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>def original(day):
    return 30
def fixed(day):
    return 90 if day &gt;= &quot;2026-07-01&quot; else 30 if day &gt;= &quot;2026-06-01&quot; else None
for fn in [original, fixed]:
    print(fn.__name__, fn(regression[&quot;as_of&quot;]) == regression[&quot;expected&quot;])
held_out = [(&quot;2026-06-15&quot;, 30), (&quot;2026-07-15&quot;, 90)]
print([fixed(day) == expected for day, expected in held_out])</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>original False
fixed True
[True, True]
</code></pre></details>

## 수업 서비스에 적용하기

시험 환경의 LangSmith 프로젝트에서 지연이 긴 요청, 예외가 난 요청, 부정 피드백을 따로 찾아보세요. 실패 실행 하나를 개발 데이터셋에 한 번만 추가하고 원문 조항으로 기대값을 확인합니다. 수정 전후 실험 링크, 계속 실패한 사례, 보관했던 질문의 최종 결과를 86일차 팀에 전달하세요. 82일차 셀로 개발 데이터셋을 먼저 준비하세요. 아래 셀은 수업용 데이터셋에 재현 문항 한 건을 작성하므로 새 문항일 때 한 번만 켜세요.

In [ ]:
run_integration = False
if run_integration:
    from langsmith import Client
    client = Client()
    dataset = client.read_dataset(dataset_name="course-product-qa-dev")
    seen = {e.inputs["as_of"] for e in client.list_examples(dataset_id=dataset.id)}
    if regression["as_of"] not in seen:
        client.create_example(inputs={"as_of": regression["as_of"]},
                              outputs={"citation":"A2", "days":regression["expected"]}, dataset_id=dataset.id)

### 제출 전 확인

시험 요청의 지연·오류·피드백을 살펴본다. 실패한 실행 한 건에 기대 동작을 붙여 평가 데이터에 추가하고, 수정 전후 결과와 별도로 보관한 질문의 최종 평가 결과를 제출한다.

[공식 문서](https://docs.langchain.com/langsmith/manage-datasets-programmatically)